In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

In [2]:
df_batch = pd.read_csv(
    "../input/gh_exp_batch_1.csv",
    dtype={"installer_name": str, "downloads": float},
    parse_dates=["date"],
)
df_batch.head()

,date,pkg,installer_name,downloads
0,2023-04-28,a-pandas-ex-df2htmlstring,requests,17.0
1,2023-04-28,a-pandas-ex-df2htmlstring,bandersnatch,79.0
2,2023-04-28,a-pandas-ex-df2htmlstring,NaN,27.0
3,2023-04-28,a-pandas-ex-df2htmlstring,Browser,37.0
4,2023-04-28,a-pandas-ex-df2htmlstring,pip,7.0


In [3]:
# Truncate to analysis window: Apr 1 - Oct 31, 2023
df_batch = df_batch.query("date <= '2023-10-31'")
print("After truncation:", df_batch.shape)
print("Date range:", df_batch["date"].min(), "to", df_batch["date"].max())

After truncation: (178190, 4)
Date range: 2023-04-19 00:00:00 to 2023-10-31 00:00:00


## Filter to human installers

Same logic as `consolidate_pypi_downloads_allinstallers.ipynb`.
Added `poetry` and `uv` which appear in the batch data (both are legitimate package managers).

In [4]:
human_installers = [
    "pip",
    "setuptools",
    "Nexus",
    "pdm",
    "Homebrew",
    "Artifactory",
    "OS",
    "Bazel",
    "pex",
    "conda",
    "chaquopy",
    "poetry",
    "uv",
]

df_dl = (
    df_batch
    .query("installer_name in @human_installers")
    .groupby(["pkg", "date"], as_index=False)["downloads"]
    .sum()
    .rename(columns={"downloads": "download_count"})
    .sort_values(["pkg", "date"], ignore_index=True)
)
print("Shape:", df_dl.shape)
df_dl.head()

Shape: (29013, 3)


,pkg,date,download_count
0,a-pandas-ex-df2htmlstring,2023-04-28,7.0
1,a-pandas-ex-df2htmlstring,2023-05-02,1.0
2,a-pandas-ex-df2htmlstring,2023-05-05,1.0
3,a-pandas-ex-df2htmlstring,2023-05-11,1.0
4,a-pandas-ex-df2htmlstring,2023-05-13,2.0


## Read treatment assignment

In [5]:
df_random_sample = (
    pd.read_csv("../input/random_sample.csv")
    .assign(treatment=1)          # ITT assignment indicator (instrument for LATE)
    .assign(boughtstars=lambda df: (df.index < 25).astype(int))
    .assign(treatment2=lambda df: df["boughtstars"] + df["treatment"])
)[["pkg", "treatment", "boughtstars", "treatment2"]]

# Compliance indicator: packages that actually received >=20 stars (endogenous for LATE)
# Pulled from original pkg_human_downloads.csv (package-level constant, time-invariant)
df_compliance = (
    pd.read_csv("../../metrics-timeseries/input/pkg_human_downloads.csv")
    .drop_duplicates("pkg")[["pkg", "treated"]]
    .assign(treated=lambda df: df["treated"].fillna(0).astype(int))
)

print("Assigned packages:", len(df_random_sample))
print("High dosage:", (df_random_sample["treatment2"] == 2).sum())
print("Low dosage:", (df_random_sample["treatment2"] == 1).sum())
print("Compliers (treated=1):", (df_compliance["treated"] == 1).sum())

Assigned packages: 100
High dosage: 25
Low dosage: 75
Compliers (treated=1): 45


## Build balanced panel and compute cumulative downloads

In [6]:
panel_start = "2023-04-01"
panel_end   = "2023-10-31"

df = (
    df_dl
    .set_index(["pkg", "date"])
    .pipe(
        lambda df_: df_.reindex(
            pd.MultiIndex.from_product(
                [
                    df_.index.get_level_values("pkg").unique(),
                    pd.date_range(panel_start, panel_end),
                ],
                names=["pkg", "date"],
            )
        )
    )
    .fillna({"download_count": 0})
    .reset_index()
    .assign(tt_downloads=lambda df: df.groupby("pkg")["download_count"].cumsum())
    # Merge ITT assignment (treatment, boughtstars, treatment2)
    .merge(df_random_sample, how="left", on="pkg", validate="m:1")
    .assign(treatment=lambda df: df["treatment"].fillna(0).astype(int))
    .assign(boughtstars=lambda df: df["boughtstars"].fillna(0).astype(int))
    .assign(treatment2=lambda df: df["treatment2"].fillna(0).astype(int))
    # Merge compliance (treated)
    .merge(df_compliance, how="left", on="pkg", validate="m:1")
    .assign(treated=lambda df: df["treated"].fillna(0).astype(int))
    .assign(date=lambda df: df["date"].dt.date)
)

# Sanity checks
assert 100 == len(df.drop_duplicates("pkg").query("treatment == 1"))
assert 25  == len(df.drop_duplicates("pkg").query("treatment2 == 2"))
assert len(df) == df["pkg"].nunique() * df["date"].nunique()

n_pkgs = df["pkg"].nunique()
n_days = df["date"].nunique()
print(f"Packages: {n_pkgs}")
print(f"Days: {n_days}  ({panel_start} to {panel_end})")
print(f"Panel rows: {len(df):,}")
print(f"Compliers: {df.drop_duplicates('pkg')['treated'].sum()}")
df.head()

Packages: 622
Days: 214  (2023-04-01 to 2023-10-31)
Panel rows: 133,108
Compliers: 45


,pkg,date,download_count,tt_downloads,treatment,boughtstars,treatment2,treated
0,a-pandas-ex-df2htmlstring,2023-04-01,0.0,0.0,0,0,0,0
1,a-pandas-ex-df2htmlstring,2023-04-02,0.0,0.0,0,0,0,0
2,a-pandas-ex-df2htmlstring,2023-04-03,0.0,0.0,0,0,0,0
3,a-pandas-ex-df2htmlstring,2023-04-04,0.0,0.0,0,0,0,0
4,a-pandas-ex-df2htmlstring,2023-04-05,0.0,0.0,0,0,0,0


In [7]:
df.to_csv("../output/pkg_human_downloads_extended.csv", index=False)